# Accent Coach — Phase 0 Validation (Experiments A–D)

Run top-to-bottom. Kernel: project `.venv` (not vendor's).

**Experiments**
- A: Synth BC voice vs RP norms → expected ≥ 85
- B: Real BC audio vs RP norms → expected ≥ 80
- C: User audio vs RP norms → expected ≤ 65 (requires owner recordings in `tts_output/accent_coach/users/owner/`)
- D: User audio vs synth BC target → expected within ±5 of C

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from accent_coach.calibration.sentences import CALIBRATION_SENTENCES, get_by_id
from accent_coach.pipeline.features import analyse_audio
from accent_coach.comparison.scoring import compare
from accent_coach.reference.rp_norms import get_rp_norms, RP_VOWEL_F1_F2_MALE
from accent_coach.models import SentenceAnalysis

print(f"Calibration set: {len(CALIBRATION_SENTENCES)} sentences loaded")


In [ ]:
## Helper: analyse a WAV + transcript and score it

def score_wav(wav_path: Path, transcript: str, sentence_id: int = 0,
              target_wav: Path | None = None) -> dict:
    from accent_coach.calibration.sentences import Sentence
    meta = get_by_id(sentence_id) if sentence_id else Sentence(
        id=0, text=transcript, sentence_type="statement", targets=[]
    )
    user_analysis = analyse_audio(wav_path, transcript, meta)
    audio, sr = sf.read(str(wav_path), always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    target_analysis = None
    if target_wav and target_wav.exists():
        target_analysis = analyse_audio(target_wav, transcript, meta)

    mean_f0 = float(np.mean([v.pitch_mean for v in user_analysis.vowels if v.pitch_mean > 70] or [120.0]))
    rp_norms = get_rp_norms(mean_f0)
    result = compare(user_analysis, audio, sr, target=target_analysis, reference_norms=rp_norms)
    return {
        "composite": result.composite_score,
        **result.skill_scores,
        "vowel_diagnostics": result.vowel_diagnostics,
        "npvi": result.rhythm_breakdown.npvi,
        "user_vowels": user_analysis.vowels,
    }


In [ ]:
## Experiment B — Real BC audio vs RP norms

# Uses existing reference clips; transcripts produced by Whisper if not on disk.
ref_clips = {
    "ref_interview": Path("../tts_output/ref_interview.wav"),
}

# Transcribe with existing Whisper (or provide manually)
def transcribe(wav_path: Path) -> str:
    import whisper
    model = whisper.load_model("base")
    result = model.transcribe(str(wav_path))
    return result["text"].strip()

exp_b_results = []
for name, wav in ref_clips.items():
    if not wav.exists():
        print(f"SKIP (missing): {wav}")
        continue
    print(f"Analysing {name}...")
    transcript = transcribe(wav)
    print(f"  Transcript: {transcript[:80]}")
    r = score_wav(wav, transcript)
    r["clip"] = name
    exp_b_results.append(r)
    print(f"  Composite: {r['composite']:.1f}")

if exp_b_results:
    print(f"\nExp B mean composite: {np.mean([r['composite'] for r in exp_b_results]):.1f} (target ≥ 80)")


In [ ]:
## Experiment C — User audio vs RP norms (BLOCKED until owner records)

owner_dir = Path("../tts_output/accent_coach/users/owner")
owner_wavs = sorted(owner_dir.glob("*.wav")) if owner_dir.exists() else []

if not owner_wavs:
    print("⚠️  Experiment C blocked: no owner recordings found.")
    print(f"   Record calibration sentences → {owner_dir}/")
    print("   Name files like: 001_please_leave.wav, 002_ship_hit.wav, ...")
else:
    exp_c_results = []
    for wav in owner_wavs:
        sid = int(wav.stem.split("_")[0])
        try:
            meta = get_by_id(sid)
        except KeyError:
            print(f"  SKIP {wav.name}: unknown sentence id {sid}")
            continue
        r = score_wav(wav, meta.text, sid)
        r["clip"] = wav.stem
        exp_c_results.append(r)
        print(f"  {wav.stem}: {r['composite']:.1f}")

    mean_c = np.mean([r["composite"] for r in exp_c_results])
    print(f"\nExp C mean composite: {mean_c:.1f} (target ≤ 65)")


In [ ]:
## Vowel scatter: user vs RP reference

def plot_vowel_scatter(user_vowels, rp_norms, title=""):
    fig, ax = plt.subplots(figsize=(8, 6))
    # RP reference centroids
    for ph, (f1, f2) in rp_norms.items():
        ax.scatter(f2, f1, marker="x", s=80, color="blue", zorder=3)
        ax.annotate(ph, (f2, f1), fontsize=8, color="blue", ha="center", va="bottom")
    # User vowels
    for v in user_vowels:
        ax.scatter(v.f2, v.f1, marker="o", s=30, alpha=0.5, color="red")
    ax.set_xlabel("F2 (Hz) →")
    ax.set_ylabel("F1 (Hz) ↓")
    ax.invert_yaxis()
    ax.invert_xaxis()
    ax.set_title(title or "Vowel space: user (red) vs RP (blue)")
    plt.tight_layout()
    plt.show()

# Run after Exp B
if exp_b_results:
    all_vowels = [v for r in exp_b_results for v in r["user_vowels"]]
    mean_f0 = float(np.mean([v.pitch_mean for v in all_vowels if v.pitch_mean > 70] or [120.0]))
    norms = get_rp_norms(mean_f0)
    plot_vowel_scatter(all_vowels, norms, "Experiment B — Real BC vs RP")


In [ ]:
## Diagnostic text dump (Exp B or C)

source_results = exp_b_results  # switch to exp_c_results when available
for r in source_results[:3]:
    print(f"\n=== {r.get('clip', '?')} ===")
    for diag in r["vowel_diagnostics"]:
        if diag.deviation_magnitude > 0.1:
            print(f"  /{diag.phoneme}/  F1={diag.user_f1:.0f} (ref {diag.target_f1:.0f})  "
                  f"F2={diag.user_f2:.0f} (ref {diag.target_f2:.0f})")
            print(f"    → {diag.articulatory_advice}")


In [ ]:
## Score summary table

import pandas as pd

rows_data = []
for label, results in [("B (real BC vs RP)", exp_b_results)]:
    if not results:
        continue
    rows_data.append({
        "Experiment": label,
        "Composite": np.mean([r["composite"] for r in results]),
        "Vowels": np.mean([r.get("vowels", 0) for r in results]),
        "Consonants": np.mean([r.get("consonants", 0) for r in results]),
        "Aspiration": np.mean([r.get("aspiration", 0) for r in results]),
        "Rhythm": np.mean([r.get("rhythm", 0) for r in results]),
        "Stress": np.mean([r.get("stress", 0) for r in results]),
        "Intonation": np.mean([r.get("intonation", 0) for r in results]),
        "nPVI": np.mean([r.get("npvi", 0) for r in results]),
    })

df = pd.DataFrame(rows_data).set_index("Experiment")
df.style.format("{:.1f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=100)
